# Explainable Anomaly Detection in IoT System Logs

**Using XGBoost, Random Forest, SMOTE, and SHAP on the BGL Supercomputer Log Dataset**

> Faisal Ahamed Syed  
> Department of Computer Information Systems and Cybersecurity  
> Auburn University at Montgomery  
> fsyed8@aum.edu

---

This notebook runs the full 7-stage pipeline:

| Stage | Description |
|-------|-------------|
| 1 | Load BGL.log and parse labels |
| 2 | Drain3 log parsing → unique templates |
| 3 | Extract 5 features |
| 4 | Stratified 80/20 split + template_freq mapping |
| 5 | Apply SMOTE to training partition only |
| 6 | Train 4 models and evaluate |
| 7 | SHAP TreeExplainer on best model |

**Runtime:** ~60–90 min on Colab free tier (full 4.7M-row dataset).

## Setup

Run the cell below to install dependencies.  
- **Google Colab:** uncomment the `!pip install` and `drive.mount` lines.  
- **Local Jupyter:** dependencies are already in `requirements.txt` — skip those two lines and set `LOG_FILE` to your local `BGL.log` path in the Configuration cell.

In [ ]:
# ── Colab only: uncomment and run ─────────────────────────────────────────
# !pip install drain3 shap imbalanced-learn xgboost -q

# from google.colab import drive
# drive.mount('/content/drive')
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import shap

warnings.filterwarnings("ignore")

## Configuration

Set `LOG_FILE` to the path of your `BGL.log` file before running.

- **Colab:** `/content/drive/MyDrive/iot-anomaly/BGL.log`
- **Local:** `"/path/to/BGL.log"` or `"BGL.log"` if in the same directory

In [ ]:
LOG_FILE     = "BGL.log"          # <-- set your path here
OUTPUT_DIR   = "../results"       # relative to this notebook
SAMPLE_SIZE  = None               # None = full dataset; or e.g. 50_000 for a quick test
SEED         = 42
TEST_SIZE    = 0.2
SHAP_SAMPLES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

---
## Stage 1 — Load and Label

Reads `BGL.log` line by line. The first field is `-` for normal entries and an alert label (e.g. `APPREAD`) for anomalies.

In [ ]:
print("[Stage 1] Loading BGL.log...")
rows = []
with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if SAMPLE_SIZE and i >= SAMPLE_SIZE:
            break
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 9:
            continue
        rows.append({
            "message": " ".join(parts[9:]),
            "label":   0 if parts[0] == "-" else 1,
        })
        if (i + 1) % 500_000 == 0:
            print(f"  Loaded {i+1:,} lines...")

df = pd.DataFrame(rows)
print(f"  Total: {len(df):,}  |  Anomaly rate: {df['label'].mean():.2%}")
df.head(3)

---
## Stage 2 — Drain3 Log Parsing

Groups raw log messages into structured templates using the Drain algorithm (`sim_th=0.4`, `depth=4`).  
Expected output: **1,823 unique templates** on the full BGL dataset.  
Runtime: ~20–30 minutes on the full 4.7M rows.

In [ ]:
print("[Stage 2] Drain3 parsing...")
config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
miner = TemplateMiner(config=config)

template_ids, template_strs = [], []
for i, msg in enumerate(df["message"]):
    res = miner.add_log_message(msg)
    template_ids.append(res["cluster_id"])
    template_strs.append(res["template_mined"])
    if (i + 1) % 500_000 == 0:
        print(f"  Parsed {i+1:,} / {len(df):,}")

df["template_id"]  = template_ids
df["template_str"] = template_strs
print(f"  Unique templates: {df['template_id'].nunique()}")

# Save checkpoint — protects against Colab runtime resets
checkpoint = os.path.join(OUTPUT_DIR, "bgl_parsed.parquet")
df.to_parquet(checkpoint, index=False)
print(f"  Checkpoint saved to {checkpoint}")

---
## Stage 3 — Feature Extraction

Five features derived from each log entry:

| Feature | Type | Description |
|---------|------|-------------|
| `template_id` | categorical | Drain3 cluster ID |
| `template_length` | discrete | Token count in template |
| `has_error` | binary | Keyword match (error/fail/fatal/exception/crash) |
| `wildcard_count` | discrete | Number of `<*>` placeholders |
| `template_freq` | continuous | Occurrence count in training set (computed in Stage 4) |

In [ ]:
print("[Stage 3] Feature extraction...")
df["template_length"] = df["template_str"].apply(lambda x: len(x.split()))
df["has_error"] = df["message"].str.contains(
    r"error|fail|fatal|exception|crash", case=False, regex=True
).astype(int)
df["wildcard_count"] = df["template_str"].str.count(r"<\*>")

print("  Features added: template_length, has_error, wildcard_count")
df[["message", "template_id", "template_length", "has_error", "wildcard_count", "label"]].head(3)

---
## Stage 4 — Train/Test Split + `template_freq`

Stratified 80/20 split. `template_freq` is computed **on the training partition only** to prevent data leakage, then mapped to the test set.

In [ ]:
print("[Stage 4] Split and template_freq...")
X_base = df[["template_id", "template_length", "has_error", "wildcard_count"]]
y      = df["label"]

X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_base, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

freq_map = X_train_b["template_id"].value_counts().to_dict()
X_train  = X_train_b.copy()
X_test   = X_test_b.copy()
X_train["template_freq"] = X_train["template_id"].map(freq_map).fillna(0).astype(int)
X_test ["template_freq"] = X_test ["template_id"].map(freq_map).fillna(0).astype(int)

print(f"  Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"  Train anomaly rate: {y_train.mean():.2%}  |  Test: {y_test.mean():.2%}")

---
## Stage 5 — SMOTE

Synthetic Minority Oversampling Technique applied **only to the training set** (`k_neighbors=5`, `random_state=42`).  
This balances the 92.66% / 7.34% class split before model training.

In [ ]:
print("[Stage 5] SMOTE...")
smote = SMOTE(k_neighbors=5, random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"  Before — Normal: {(y_train==0).sum():,}  Anomaly: {(y_train==1).sum():,}")
print(f"  After  — Normal: {(y_train_sm==0).sum():,}  Anomaly: {(y_train_sm==1).sum():,}")

---
## Stage 6 — Train 4 Models

Four configurations: Random Forest and XGBoost, each with and without SMOTE.  
Hyperparameters match Nguyen & Nguyen (2023) for direct comparability.

In [ ]:
print("[Stage 6] Training 4 models...")

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    print(f"  {name}...")
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_te, pred),  4),
        "Precision": round(precision_score(y_te, pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_te, pred,    zero_division=0), 4),
        "F1":        round(f1_score(y_te, pred,        zero_division=0), 4),
    }

results = [
    evaluate("RF  (no SMOTE)",
             RandomForestClassifier(n_estimators=20, random_state=SEED, n_jobs=-1),
             X_train, y_train, X_test, y_test),
    evaluate("RF  (SMOTE)",
             RandomForestClassifier(n_estimators=20, random_state=SEED, n_jobs=-1),
             X_train_sm, y_train_sm, X_test, y_test),
    evaluate("XGB (no SMOTE)",
             XGBClassifier(learning_rate=0.3, max_depth=6, n_estimators=100,
                           random_state=SEED, eval_metric="logloss", n_jobs=-1),
             X_train, y_train, X_test, y_test),
    evaluate("XGB (SMOTE)",
             XGBClassifier(learning_rate=0.3, max_depth=6, n_estimators=100,
                           random_state=SEED, eval_metric="logloss", n_jobs=-1),
             X_train_sm, y_train_sm, X_test, y_test),
]

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "results.csv"), index=False)
print()
results_df

---
## Stage 7 — SHAP Analysis

TreeExplainer on the best model (XGB + SMOTE) over a stratified 5,000-entry test sample.  
Produces three figures saved to `results/`:
- `shap_importance.png` — global feature importance bar chart
- `shap_beeswarm.png` — directional beeswarm plot
- `shap_waterfall.png` — single anomalous instance explanation

In [ ]:
print("[Stage 7] SHAP analysis...")
features = ["template_id", "template_length", "has_error",
            "wildcard_count", "template_freq"]

best = XGBClassifier(learning_rate=0.3, max_depth=6, n_estimators=100,
                     random_state=SEED, eval_metric="logloss", n_jobs=-1)
best.fit(X_train_sm, y_train_sm)

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_test), size=SHAP_SAMPLES, replace=False)
X_sample = X_test.iloc[sample_idx]
y_sample = y_test.iloc[sample_idx]

explainer   = shap.TreeExplainer(best)
shap_values = explainer.shap_values(X_sample)

# Plot 1 — global bar
shap.summary_plot(shap_values, X_sample, feature_names=features,
                  plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_importance.png"), dpi=150)
plt.show()

# Plot 2 — beeswarm
shap.summary_plot(shap_values, X_sample, feature_names=features, show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_beeswarm.png"), dpi=150)
plt.show()

# Plot 3 — waterfall (one anomalous instance)
anomaly_idx = (y_sample.values == 1).nonzero()[0][0]
shap.plots.waterfall(
    shap.Explanation(
        values        = shap_values[anomaly_idx],
        base_values   = explainer.expected_value,
        data          = X_sample.iloc[anomaly_idx].values,
        feature_names = features
    ),
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_waterfall.png"), dpi=150)
plt.show()

print(f"\nDone. Results saved to {os.path.abspath(OUTPUT_DIR)}/")